# Automatic History Matching Workflow

This notebook demonstrates the fully automated history matching workflow using the new object-oriented API. We'll show how to set up and run complete multi-iteration workflows with minimal user intervention.

## Overview

The automatic workflow features:
- **Automatic Feature Selection**: Algorithm chooses optimal features for emulation
- **Multi-Iteration Execution**: Run complete workflows with a single command
- **Convergence Detection**: Automatic stopping when parameter space is sufficiently constrained
- **Progress Monitoring**: Built-in callbacks and progress tracking
- **Adaptive Strategies**: Dynamic adjustment of parameters based on performance

This is ideal for production workflows and large-scale parameter estimation problems.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time

# Import the modern history matching API
import history_matching as hm

# Configure matplotlib for notebook
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## SIR Model for Automatic Workflow

We'll use the same SIR epidemiological model as in previous examples, but configure it for fully automated execution with multiple iterations.

In [ ]:
def sir_model(samples_df):
    """
    SIR epidemiological model simulation.
    
    Parameters:
    - beta: transmission rate
    - gamma: recovery rate
    - initial_infected: initial number of infected individuals
    
    Returns DataFrame with time series and summary statistics.
    """
    results = []
    
    for idx, row in samples_df.iterrows():
        beta = row['beta']
        gamma = row['gamma']
        initial_infected = row['initial_infected']
        
        # Model parameters
        N = 1000  # Total population
        dt = 0.1  # Time step
        T = 100   # Total time
        steps = int(T / dt)
        
        # Initialize
        S = np.zeros(steps)
        I = np.zeros(steps)
        R = np.zeros(steps)
        
        S[0] = N - initial_infected
        I[0] = initial_infected
        R[0] = 0
        
        # Simulate
        for t in range(1, steps):
            dS = -beta * S[t-1] * I[t-1] / N
            dI = beta * S[t-1] * I[t-1] / N - gamma * I[t-1]
            dR = gamma * I[t-1]
            
            S[t] = S[t-1] + dS * dt
            I[t] = I[t-1] + dI * dt
            R[t] = R[t-1] + dR * dt
        
        # Calculate multiple summary statistics for automatic feature selection
        peak_infected = np.max(I)
        time_to_peak = np.argmax(I) * dt
        final_recovered = R[-1]
        total_infected = np.max(R)  # Maximum recovered = total who got infected
        infected_at_day_30 = I[int(30/dt)] if int(30/dt) < len(I) else I[-1]
        infected_at_day_60 = I[int(60/dt)] if int(60/dt) < len(I) else I[-1]
        attack_rate = total_infected / N
        
        # Additional derived features
        R0_effective = beta / gamma  # Basic reproduction number
        duration_above_10pct = np.sum(I > 0.1 * peak_infected) * dt
        area_under_curve = np.trapz(I, dx=dt)
        
        result = {
            'peak_infected': peak_infected,
            'time_to_peak': time_to_peak,
            'final_recovered': final_recovered,
            'total_infected': total_infected,
            'infected_day_30': infected_at_day_30,
            'infected_day_60': infected_at_day_60,
            'attack_rate': attack_rate,
            'R0_effective': R0_effective,
            'duration_above_10pct': duration_above_10pct,
            'area_under_curve': area_under_curve,
        }
        
        results.append(result)
    
    return pd.DataFrame(results)

## Setup for Automatic Workflow

We'll create a comprehensive setup with multiple features available for automatic selection.

In [ ]:
# Create parameter space with wider ranges for challenging calibration
parameter_bounds = {
    'beta': (0.1, 1.0),
    'gamma': (0.05, 0.5), 
    'initial_infected': (1, 20)
}

parameter_space = hm.ParameterSpace(parameter_bounds)

print(f"Parameter space defined with {len(parameter_space.get_parameter_names())} parameters")
for name in parameter_space.get_parameter_names():
    lo, hi = parameter_space.get_bounds(name)
    print(f"  {name}: [{lo}, {hi}]")

In [ ]:
# Create synthetic "observations" with uncertainty
np.random.seed(42)

# True parameter values (hidden from the algorithm)
true_beta = 0.3
true_gamma = 0.1
true_initial = 5

# Generate true data
true_params = pd.DataFrame({
    'beta': [true_beta],
    'gamma': [true_gamma], 
    'initial_infected': [true_initial]
})

true_outputs = sir_model(true_params)
print("True model outputs:")
print(true_outputs.iloc[0])

In [ ]:
# Create observations with uncertainty for multiple features
observation_data = {}

# Add observations for several key features with different uncertainty levels
observation_noise = 0.05  # 5% relative noise

# Peak infected (most reliable observation)
observation_data['peak_infected'] = (
    true_outputs.iloc[0]['peak_infected'],
    observation_noise * true_outputs.iloc[0]['peak_infected']
)

# Time to peak (moderate uncertainty) 
observation_data['time_to_peak'] = (
    true_outputs.iloc[0]['time_to_peak'],
    2 * observation_noise * true_outputs.iloc[0]['time_to_peak']
)

# Final recovered (high uncertainty due to long-term measurement)
observation_data['final_recovered'] = (
    true_outputs.iloc[0]['final_recovered'],
    3 * observation_noise * true_outputs.iloc[0]['final_recovered']
)

# Attack rate (derived quantity, moderate uncertainty)
observation_data['attack_rate'] = (
    true_outputs.iloc[0]['attack_rate'],
    1.5 * observation_noise * true_outputs.iloc[0]['attack_rate']
)

observations = hm.ObservationData(observation_data)

print(f"\nObservations created for {len(observations.get_feature_names())} features:")
for feature in observations.get_feature_names():
    mean, std = observations.get_target_for_feature(feature)
    print(f"  {feature}: {mean:.3f} +/- {std:.3f}")

## Advanced Builder Configuration

We'll configure the builder for automatic workflow with sophisticated strategies.

In [ ]:
# Create advanced builder with automatic feature selection
builder = hm.HistoryMatchingBuilder.from_data(
    parameter_bounds=parameter_bounds,
    observations=observation_data
)

# Configure for automatic workflow
builder = (builder
    .with_sampling_strategy({'type': 'lhs', 'criterion': 'maximin'})  # High-quality LHS sampling
    .with_feature_selection({'method': 'fano', 'max_features': 3})  # Automatic selection
    .with_emulator_type('linear')  # Use linear emulators for speed
    .with_samples_per_iteration(500)  # More samples for robustness
    .with_max_iterations(5)  # Allow multiple iterations
    .with_implausibility_threshold(3.0)  # Standard threshold
    .with_oversample_factor(5.0)  # Higher oversampling for filtering
    .with_space_reduction(False)  # Keep original space for stability
    .with_random_seed(123)  # For reproducibility
)

print("Advanced builder configured for automatic workflow")
config = builder.preview_configuration()
for key, value in config.items():
    print(f"  {key}: {value}")

## Progress Monitoring Setup

Let's set up comprehensive progress monitoring and callbacks for the automatic workflow.

In [ ]:
# Progress tracking variables
iteration_results = []
progress_log = []
feature_selection_history = []

def iteration_callback(result):
    """Called after each completed iteration."""
    iteration_results.append(result)
    
    # Log feature selection
    selected_features = result.selected_features
    feature_selection_history.append(selected_features)
    
    print(f"\n=== Iteration {result.iteration} Complete ===")
    print(f"Selected features: {selected_features}")
    print(f"Samples generated: {len(result.samples)}")
    print(f"Emulators trained: {len(result.emulators)}")
    print(f"Non-implausible fraction: {result.non_implausible_fraction:.3f}")

def progress_callback(progress):
    """Called on progress updates."""
    progress_entry = {
        'iteration': progress.current_iteration,
        'total_samples': progress.total_samples_generated,
        'accepted_samples': progress.total_samples_accepted,
        'acceptance_rate': progress.acceptance_rate,
        'emulators_trained': progress.total_emulators_trained
    }
    progress_log.append(progress_entry)

print("Progress monitoring callbacks defined")

## Execute Automatic Workflow

Now we'll build the engine and execute the complete automatic workflow.

In [ ]:
# Build the engine
print("Building history matching engine...")
engine = builder.build()

# Set up the simulation function
engine.set_simulation_function(sir_model)

# Add our monitoring callbacks
engine.add_iteration_callback(iteration_callback)
engine.add_progress_callback(progress_callback)

print(f"Engine built successfully: {engine}")
print(f"Engine state: {engine.state}")

In [ ]:
# Execute the complete automatic workflow
print("Starting automatic history matching workflow...\n")
start_time = time.time()

try:
    # This will run until convergence or max iterations
    results = engine.run(auto_commit=True)
    
    end_time = time.time()
    total_time = end_time - start_time
    
    print(f"\n{'='*50}")
    print("AUTOMATIC WORKFLOW COMPLETED SUCCESSFULLY")
    print(f"{'='*50}")
    print(f"Total iterations: {len(results)}")
    print(f"Total time: {total_time:.2f} seconds")
    print(f"Final engine state: {engine.state}")
    print(f"Final acceptance rate: {engine.acceptance_rate:.4f}")
    
except Exception as e:
    print(f"Workflow failed: {e}")
    raise

## Automatic Workflow Analysis

Let's analyze the results of our automatic workflow.

In [ ]:
# Overall workflow statistics
if progress_log:
    final_progress = progress_log[-1]
    
    print("=== Workflow Statistics ===")
    print(f"Total samples generated: {final_progress['total_samples']}")
    print(f"Total samples accepted: {final_progress['accepted_samples']}")
    print(f"Overall acceptance rate: {final_progress['acceptance_rate']:.4f}")
    print(f"Total emulators trained: {final_progress['emulators_trained']}")
    print(f"Average samples per iteration: {final_progress['total_samples'] / len(results):.1f}")
    
    # Efficiency metrics
    samples_per_second = final_progress['total_samples'] / total_time
    print(f"\n=== Performance Metrics ===")
    print(f"Samples per second: {samples_per_second:.1f}")
    print(f"Time per iteration: {total_time / len(results):.2f} seconds")
    print(f"Samples per emulator: {final_progress['total_samples'] / max(final_progress['emulators_trained'], 1):.1f}")

In [ ]:
# Feature selection analysis
print("=== Automatic Feature Selection History ===")
all_features = set()
for i, features in enumerate(feature_selection_history, 1):
    print(f"Iteration {i}: {features}")
    all_features.update(features)

# Feature usage frequency
feature_usage = {}
for features in feature_selection_history:
    for feature in features:
        feature_usage[feature] = feature_usage.get(feature, 0) + 1

print(f"\n=== Feature Usage Summary ===")
print(f"Total unique features used: {len(all_features)}")
print("Feature usage frequency:")
for feature, count in sorted(feature_usage.items(), key=lambda x: x[1], reverse=True):
    print(f"  {feature}: {count}/{len(feature_selection_history)} iterations ({100*count/len(feature_selection_history):.1f}%)")

In [ ]:
# Parameter space analysis via non-implausible fractions
print("=== Non-Implausible Fraction by Iteration ===")

for i, result in enumerate(iteration_results, 1):
    print(f"  Iteration {i}: {result.non_implausible_fraction:.4f} non-implausible fraction")

# Show parameter ranges from final accepted samples
if results:
    final_samples = results[-1].non_implausible_points
    print(f"\nFinal non-implausible parameter ranges (from {len(final_samples)} points):")
    for param in parameter_space.get_parameter_names():
        if param in final_samples.columns:
            lo, hi = final_samples[param].min(), final_samples[param].max()
            orig_lo, orig_hi = parameter_space.get_bounds(param)
            reduction = 1 - (hi - lo) / (orig_hi - orig_lo)
            print(f"  {param}: [{lo:.3f}, {hi:.3f}] ({reduction:.1%} reduction from original)")

## Visualization of Automatic Workflow Results

Let's create comprehensive visualizations of our automatic workflow.

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Automatic History Matching Workflow Results', fontsize=16, fontweight='bold')

# 1. Acceptance rate evolution
iterations = [p['iteration'] for p in progress_log]
acceptance_rates = [p['acceptance_rate'] for p in progress_log]

axes[0,0].plot(iterations, acceptance_rates, 'bo-', linewidth=2, markersize=8)
axes[0,0].axhline(y=0.01, color='red', linestyle='--', alpha=0.7, label='Convergence threshold')
axes[0,0].set_xlabel('Iteration')
axes[0,0].set_ylabel('Acceptance Rate')
axes[0,0].set_title('Acceptance Rate Evolution')
axes[0,0].grid(True, alpha=0.3)
axes[0,0].legend()
axes[0,0].set_yscale('log')

# 2. Non-implausible fraction by iteration
nif_values = [r.non_implausible_fraction for r in iteration_results]
axes[0,1].plot(range(1, len(nif_values)+1), nif_values, 'go-', linewidth=2, markersize=6)
axes[0,1].set_xlabel('Iteration')
axes[0,1].set_ylabel('Non-Implausible Fraction')
axes[0,1].set_title('Space Reduction Progress')
axes[0,1].grid(True, alpha=0.3)

# 3. Feature selection frequency
feature_usage = {}
for features in feature_selection_history:
    for feature in features:
        feature_usage[feature] = feature_usage.get(feature, 0) + 1

features = list(feature_usage.keys())
frequencies = [feature_usage[f] for f in features]
colors_bar = plt.cm.Set3(np.linspace(0, 1, len(features)))

bars = axes[0,2].bar(range(len(features)), frequencies, color=colors_bar)
axes[0,2].set_xlabel('Features')
axes[0,2].set_ylabel('Usage Count')
axes[0,2].set_title('Feature Selection Frequency')
axes[0,2].set_xticks(range(len(features)))
axes[0,2].set_xticklabels(features, rotation=45, ha='right')
axes[0,2].grid(True, alpha=0.3)

for bar, freq in zip(bars, frequencies):
    height = bar.get_height()
    axes[0,2].text(bar.get_x() + bar.get_width()/2., height + 0.05,
                   f'{freq}', ha='center', va='bottom')

# 4. Sample generation over time
total_samples = [p['total_samples'] for p in progress_log]
accepted_samples = [p['accepted_samples'] for p in progress_log]

axes[1,0].plot(iterations, total_samples, 'b-', label='Total generated', linewidth=2)
axes[1,0].plot(iterations, accepted_samples, 'g-', label='Accepted', linewidth=2)
axes[1,0].fill_between(iterations, accepted_samples, alpha=0.3, color='green')
axes[1,0].set_xlabel('Iteration')
axes[1,0].set_ylabel('Cumulative Samples')
axes[1,0].set_title('Sample Generation Progress')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# 5. Final parameter estimates
if results:
    final_result = results[-1]
    final_samples = final_result.samples
    
    param_data = []
    param_labels = []
    param_true_values = []
    
    for param in ['beta', 'gamma', 'initial_infected']:
        param_data.append(final_samples[param])
        param_labels.append(param)
        
        if param == 'beta':
            param_true_values.append(true_beta)
        elif param == 'gamma':
            param_true_values.append(true_gamma)
        else:
            param_true_values.append(true_initial)
    
    bp = axes[1,1].boxplot(param_data, labels=param_labels, patch_artist=True)
    
    colors_box = ['lightblue', 'lightgreen', 'lightcoral']
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
    
    for i, (true_val, param) in enumerate(zip(param_true_values, param_labels)):
        axes[1,1].scatter(i+1, true_val, color='red', s=100, marker='*', 
                         label='True value' if i == 0 else '', zorder=5)
    
    axes[1,1].set_ylabel('Parameter Value')
    axes[1,1].set_title('Final Parameter Distributions')
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)

# 6. Workflow efficiency metrics
efficiency_metrics = {
    'Samples/sec': samples_per_second,
    'Time/iter (s)': total_time / len(results),
    'Features/iter': np.mean([len(f) for f in feature_selection_history]),
}

metric_names = list(efficiency_metrics.keys())
metric_values = list(efficiency_metrics.values())

bars = axes[1,2].bar(range(len(metric_names)), metric_values, 
                     color=['skyblue', 'lightgreen', 'orange'])
axes[1,2].set_xlabel('Metrics')
axes[1,2].set_ylabel('Value')
axes[1,2].set_title('Workflow Efficiency Metrics')
axes[1,2].set_xticks(range(len(metric_names)))
axes[1,2].set_xticklabels(metric_names, rotation=45, ha='right')
axes[1,2].grid(True, alpha=0.3)

for bar, value in zip(bars, metric_values):
    height = bar.get_height()
    axes[1,2].text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                   f'{value:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## Parameter Estimation Quality Assessment

Let's assess how well our automatic workflow recovered the true parameters.

In [ ]:
# Assess parameter estimation quality
if results:
    final_samples = results[-1].samples
    
    print("=== Parameter Estimation Assessment ===")
    print(f"Final sample size: {len(final_samples)}")
    print()
    
    # True values
    true_values = {
        'beta': true_beta,
        'gamma': true_gamma,
        'initial_infected': true_initial
    }
    
    # Calculate statistics for each parameter
    for param in ['beta', 'gamma', 'initial_infected']:
        values = final_samples[param]
        true_val = true_values[param]
        
        mean_est = np.mean(values)
        std_est = np.std(values)
        median_est = np.median(values)
        q25 = np.percentile(values, 25)
        q75 = np.percentile(values, 75)
        
        # Error metrics
        mean_error = abs(mean_est - true_val)
        relative_error = 100 * mean_error / true_val
        
        # Coverage: is true value within range?
        min_val, max_val = np.min(values), np.max(values)
        covered = min_val <= true_val <= max_val
        
        print(f"{param.upper()}:")
        print(f"  True value: {true_val:.3f}")
        print(f"  Estimated: {mean_est:.3f} ± {std_est:.3f}")
        print(f"  Median [Q25, Q75]: {median_est:.3f} [{q25:.3f}, {q75:.3f}]")
        print(f"  Range: [{min_val:.3f}, {max_val:.3f}]")
        print(f"  Absolute error: {mean_error:.3f}")
        print(f"  Relative error: {relative_error:.1f}%")
        print(f"  True value covered: {'✓' if covered else '✗'}")
        print()
    
    # Overall assessment
    all_relative_errors = []
    all_covered = []
    
    for param in ['beta', 'gamma', 'initial_infected']:
        values = final_samples[param]
        true_val = true_values[param]
        mean_est = np.mean(values)
        relative_error = 100 * abs(mean_est - true_val) / true_val
        all_relative_errors.append(relative_error)
        
        min_val, max_val = np.min(values), np.max(values)
        covered = min_val <= true_val <= max_val
        all_covered.append(covered)
    
    avg_relative_error = np.mean(all_relative_errors)
    coverage_rate = 100 * np.mean(all_covered)
    
    print("=== Overall Assessment ===")
    print(f"Average relative error: {avg_relative_error:.1f}%")
    print(f"Parameter coverage rate: {coverage_rate:.0f}%")
    
    if avg_relative_error < 10 and coverage_rate == 100:
        print(" EXCELLENT: High accuracy and complete coverage!")
    elif avg_relative_error < 20 and coverage_rate >= 66:
        print("✓ GOOD: Reasonable accuracy and coverage")
    else:
        print(" NEEDS IMPROVEMENT: Consider more iterations or different strategies")

## Workflow Configuration Summary

Let's summarize the configuration that led to these results.

In [ ]:
print("=== Automatic Workflow Configuration Summary ===")
print()
print("PARAMETER SPACE:")
print(f"  Parameters: {len(parameter_space.get_parameter_names())}")
for param in parameter_space.get_parameter_names():
    lo, hi = parameter_space.get_bounds(param)
    print(f"  {param}: [{lo}, {hi}]")

print("\nOBSERVATIONS:")
print(f"  Features observed: {len(observations.get_feature_names())}")
for feature in observations.get_feature_names():
    mean, std = observations.get_target_for_feature(feature)
    uncertainty = 100 * std / mean if mean != 0 else 0
    print(f"  {feature}: {uncertainty:.1f}% uncertainty")

print("\nSTRATEGY CONFIGURATION:")
config = builder.preview_configuration()
for key, value in config.items():
    print(f"  {key}: {value}")

print("\nWORKFLOW RESULTS:")
print(f"  Iterations completed: {len(results)}")
print(f"  Total execution time: {total_time:.2f} seconds")
print(f"  Final acceptance rate: {engine.acceptance_rate:.4f}")

print("\n" + "="*60)
print("AUTOMATIC WORKFLOW DEMONSTRATION COMPLETE")
print("="*60)

## Key Takeaways

This notebook demonstrated the powerful automatic workflow capabilities of the history matching library:

### Automatic Features:
- **Automatic Feature Selection**: The algorithm intelligently chose the most informative features for emulation
- **Multi-Iteration Execution**: Complete workflow with a single `engine.run()` call
- **Convergence Detection**: Automatic stopping when acceptance rate drops below threshold
- **Parameter Space Reduction**: Automatic constraint of parameter space based on implausibility

### Monitoring and Analysis:
- **Real-time Progress Tracking**: Comprehensive callbacks and logging
- **Performance Metrics**: Detailed efficiency and quality assessments
- **Rich Visualizations**: Multi-panel analysis of workflow results
- **Quality Assessment**: Automatic evaluation of parameter estimation accuracy

### Production Readiness:
- **Robust Configuration**: Advanced builder with sophisticated strategies
- **Error Handling**: Graceful handling of edge cases and failures
- **Scalability**: Efficient execution suitable for large parameter spaces
- **Reproducibility**: Seed-based reproducible workflows

This automatic workflow approach is ideal for:
- **Production parameter estimation pipelines**
- **Large-scale uncertainty quantification studies**
- **Automated model calibration systems**
- **Research workflows requiring minimal manual intervention**